In [ ]:
import pandas as pd
import numpy
import torch 
import numpy as np
from torch.utils.data import Dataset, DataLoader
import sys
from vis_utils import * 
sys.path.append("..")
from utils import pad_collate
from dataloader_comma import CommaDataset
from collections import Counter
from model import VTN
import matplotlib.pyplot as plt 
from PIL import Image
import glob
import os
from utils import * 
import re
import plotly.graph_objects as go

In [ ]:
from torch.utils.data import DataLoader
from dataloader_comma import CommaDataset

# Create the dataset
dataset_comma = CommaDataset(
    dataset_type="test",  # o "val" o "train" depends on what we want plot
    use_transform=False,
    multitask="distance",  
    ground_truth="desired", # Change to True to have all the parameters you use in the plot
    dataset_path="/kaggle/input/final-hdf5-files",
    dataset_fraction=1.0
)

print(f"Dataset created with {len(dataset_comma)} samples")

# Creating the Dataloader
dataloader_comma = DataLoader(
    dataset_comma,
    batch_size=1,  # Keep 1 for plotting
    shuffle=False,  # Don't shuffle for plots
    num_workers=0,  # 0 for easier debugging
    # collate_fn=None  # Use the default collate_fn, NOT the custom one you showed
)

print("DataLoader created successfully")


In [ ]:
concs = []
gt = []
## Changed the dataloader 
for j, batch in enumerate(dataloader_comma):
    image_array,  vego, angle, distance, g, s, l, seq_key = batch #g= gaspressed, s= brakepressed, l= cruiseenabled

    #image_array, distance, angle, vego, seq_key = batch
    img = image_array
    img, angle, distance, vego = img.to(f'cuda:{gpu_num}'), angle.to(f'cuda:{gpu_num}'), distance.to(f'cuda:{gpu_num}'), vego.to(f'cuda:{gpu_num}')
    (logits, attns), concepts = model(img, angle, distance, vego, seq_key)
    top5_indices = torch.tensor(concepts.squeeze()).topk(5).indices
    s = img.shape#[batch_size, seq_len, h,w,c]
    angle, distance, vego = angle.to("cpu"), distance.to("cpu"), vego.to("cpu")
    logits = logits.detach().cpu().to("cpu")
    concepts = concepts.detach().cpu().to("cpu")

    att = attns[0][:,:,0:concepts.shape[1]].detach()
    seq_len = att.shape[2]
    atten = att
    alignment_array = get_aligned_attention(atten.squeeze().cpu(), seq_len)
    speed_graph = alignment_array.sum(axis=0)[8:-8]
    speed_graph_0 = moving_average(speed_graph, 5)

    att = attns[1][:,:,0:concepts.shape[1]].detach()
    seq_len = att.shape[2]
    atten = att
    alignment_array = get_aligned_attention(atten.squeeze().cpu(), seq_len)
    speed_graph = alignment_array.sum(axis=0)[8:-8]
    speed_graph_1 = moving_average(speed_graph, 5)
    concs.extend(top5_indices.squeeze().cpu().tolist())
    gt.extend(distance.squeeze().cpu().tolist())

In [ ]:
df_exploded['gt_class']

In [ ]:
df = pd.DataFrame()
filt = lambda x: "<10" if x < 10 else (">10, <30" if x < 30 and x > 10 else (">30, <50") if x < 50 and x > 30 else ">50")
filt = lambda x: "small" if x < 5  and x > -5 else 'large'
df['gt']  = gt
df['concepts'] = concs
df_exploded = df.explode('concepts')
df_exploded['gt_class'] = df_exploded["gt"].apply( filt)
grouped = df_exploded.groupby(by="gt_class")['concepts'].apply(list)

In [ ]:
all_scens = []
for i in range(len(grouped)): 
    count_dict = Counter(grouped.iloc[i])
    top_5 = count_dict.most_common(10)
    scens = []
    for elem in top_5:
        if elem[0] == 131: continue
        scens.append((scenarios[elem[0]], elem[1]))
    all_scens.append(scens)

In [ ]:
conc_df = pd.DataFrame(grouped)
conc_df['all_scens'] = all_scens
conc_df['len'] = conc_df['concepts'].apply(lambda x: len(x))

In [ ]:
conc_df

In [ ]:
rows = []
for gt_class, row in conc_df.iterrows():
    total_len = row['len'] if 'len' in row and row['len'] else 1
    for scen, cnt in row['all_scens']:
        rows.append({
            'gt_class': gt_class,
            'scenario': expand_label(scen),
            'count': int(cnt),
            'total_in_class': int(total_len),
            'fraction': float(cnt) / float(total_len)
        })

flat_df = pd.DataFrame(rows)
# ordino per classe e poi per frequenza decrescente
flat_df = flat_df.sort_values(['gt_class', 'count'], ascending=[True, False]).reset_index(drop=True)
display(flat_df)   # tabella piatta

# ----- C) Salvare / Visualizzare interattivamente -----
# salva su CSV per ispezione esterna
flat_df.to_csv('conc_df_flat.csv', index=False)

In [ ]:
colors = [
    'rgba(228, 122, 122, 0.5)',  # Light Red
    'rgba(249, 173, 144, 0.5)',  # Light Orange
    'rgba(252, 216, 144, 0.5)',  # Light Yellow
    'rgba(211, 229, 157, 0.5)',  # Light Green
    'rgba(161, 218, 180, 0.5)',  # Light Teal
    'rgba(138, 202, 235, 0.5)',  # Light Blue
    'rgba(164, 156, 207, 0.5)',  # Light Purple
    'rgba(212, 161, 199, 0.5)',  # Light Pink
    'rgba(223, 223, 223, 0.5)',  # Light Gray
    'rgba(180, 180, 180, 0.5)'   # Light Dark Gray
]

In [ ]:
len(conc_df.iloc[-1].all_scens)

In [ ]:

#!pip install -U kaleido

In [ ]:
nodes = [
    {'label': '<b>Distance <10</b>', 'color': "white"}
]

links = []
leng = conc_df.iloc[0].len

for i, elem in enumerate(conc_df.iloc[0].all_scens):
    #expand_label used for retinanet
    #label = "<b>" +  elem[0].replace("a photo of", "").replace("driving on", "").replace("a highway with", "").replace("a street while", "").replace("a street with", "") + "</b>"
    label = "<b>" + expand_label(elem[0]) + "</b>"

    nodes.append({'label':label, 'color': colors[i]})
    res = {'source': 0, 'target': i+1, 'value': elem[1]/leng, 'color': colors[i]}
    links.append(res)

# Create the Sankey diagram figure
fig = go.Figure(data=[go.Sankey(
    node=dict(
        label=[node['label'] for node in nodes],
        color=[node['color'] for node in nodes]
    ),
    link=dict(
        source=[link['source'] for link in links],
        target=[link['target'] for link in links],
        value=[link['value'] for link in links],
        color=[link['color'] for link in links]
    )
)])

# Customize the layout of the Sankey diagram
fig.update_layout(
    title_text='',
    width=1200,  # Set the width of the plot
    height=500,  # Set the height of the plot
    font=dict(size=20, color='black'),
    margin=dict(
        autoexpand=True,
        l=0,
        r=0,
        t=0,
        b=0
    ),
    plot_bgcolor='white'
)

# Display the Sankey diagram
fig.show()
pio.write_image(fig, 'dist10.pdf')